In [1]:
#!pip install datasets evaluate --upgrade
#!python -m spacy download fr_core_news_sm

# Preparing Data

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import tqdm
import evaluate
import pandas as pd
import csv
import time

from collections import Counter
from torch.utils.data import Dataset

In [3]:
seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [4]:
!unzip data.zip

Archive:  data.zip
replace data/eng-fra.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [5]:
def load_and_clean_data(file_path):
    df = pd.read_csv(file_path, sep='\t', names=['src', 'trg'], engine='python', quoting=csv.QUOTE_NONE)
    df['src'] = df['src'].str.strip()
    df['trg'] = df['trg'].str.strip()
    return df

In [6]:
df = load_and_clean_data('/content/data/eng-fra.txt')
print(df.head())

     src         trg
0    Go.        Va !
1   Run!     Cours !
2   Run!    Courez !
3   Wow!  Ça alors !
4  Fire!    Au feu !


In [7]:
class TranslationDataset(Dataset):
    def __init__(self, df):
        self.examples = [
            {"en": row["src"], "fr": row["trg"]}
            for _, row in df.iterrows()
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

# Split your dataframe into train, val, test
train_df = df.sample(frac=0.8, random_state=42)
remaining_df = df.drop(train_df.index)
val_df = remaining_df.sample(frac=0.5, random_state=42)
test_df = remaining_df.drop(val_df.index)

In [8]:
train_data = TranslationDataset(train_df)
validation_data = TranslationDataset(val_df)
test_data = TranslationDataset(test_df)

In [9]:
train_data[0]

{'en': 'The wind was so strong, we were nearly blown off the road.',
 'fr': 'Le vent était tellement fort que nous avons presque été poussés en dehors de la route.'}

In [10]:
validation_data[0]

{'en': "Don't cross this bridge.", 'fr': 'Ne traversez pas ce pont.'}

In [11]:
test_data[0]

{'en': 'Help!', 'fr': "À l'aide\u202f!"}

## Tokenizers

In [12]:
en_nlp = spacy.load("en_core_web_sm")
fr_nlp = spacy.load("fr_core_news_sm")

In [13]:
string = "What a lovely day it is today!"
[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

In [14]:
def tokenize_example(example, en_nlp, fr_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    fr_tokens = [token.text for token in fr_nlp.tokenizer(example["fr"])][:max_length]

    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        fr_tokens = [token.lower() for token in fr_tokens]

    en_tokens = [sos_token] + en_tokens + [eos_token]
    fr_tokens = [sos_token] + fr_tokens + [eos_token]

    example["en_tokens"] = en_tokens
    example["fr_tokens"] = fr_tokens

    return example

In [15]:
max_length = 1000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "fr_nlp": fr_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

In [16]:
train_data.examples = [tokenize_example(ex, **fn_kwargs) for ex in train_data.examples]
validation_data.examples = [tokenize_example(ex, **fn_kwargs) for ex in validation_data.examples]
test_data.examples  = [tokenize_example(ex, **fn_kwargs) for ex in test_data.examples]

In [17]:
train_data[0]

{'en': 'The wind was so strong, we were nearly blown off the road.',
 'fr': 'Le vent était tellement fort que nous avons presque été poussés en dehors de la route.',
 'en_tokens': ['<sos>',
  'the',
  'wind',
  'was',
  'so',
  'strong',
  ',',
  'we',
  'were',
  'nearly',
  'blown',
  'off',
  'the',
  'road',
  '.',
  '<eos>'],
 'fr_tokens': ['<sos>',
  'le',
  'vent',
  'était',
  'tellement',
  'fort',
  'que',
  'nous',
  'avons',
  'presque',
  'été',
  'poussés',
  'en',
  'dehors',
  'de',
  'la',
  'route',
  '.',
  '<eos>']}

## Vocabularies

In [18]:
class Vocabulary:
    def __init__(self, special_tokens, unk_token="<unk>"):
        self.special_tokens = special_tokens
        self.unk_token = unk_token
        self.stoi = {}
        self.itos = {}

        for idx, token in enumerate(self.special_tokens):
            self.stoi[token] = idx
            self.itos[idx] = token

        self.unk_idx = self.stoi[self.unk_token]

    def build_vocab(self, tokenized_texts, min_freq):
        counter = Counter()
        for tokens in tokenized_texts:
            counter.update(tokens)

        idx = len(self.stoi)
        for word, freq in counter.items():
            if freq >= min_freq and word not in self.stoi:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def get_itos(self):
        return [self.itos[i] for i in range(len(self.itos))]

    def get_stoi(self):
        return self.stoi

    def __len__(self):
        return len(self.stoi)

    def __getitem__(self, token):
        if isinstance(token, list):
            return [self.stoi.get(t, self.unk_idx) for t in token]
        return self.stoi.get(token, self.unk_idx)

    def __call__(self, tokens):
        return self.__getitem__(tokens)

    def set_default_index(self, index):
        self.unk_idx = index

    def lookup_indices(self, tokens):
        return [self.stoi.get(token, self.unk_idx) for token in tokens]

    def lookup_tokens(self, indices):
        return [self.itos.get(idx, self.unk_token) for idx in indices]

In [19]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"
special_tokens = [unk_token, pad_token, sos_token, eos_token]

all_en_tokens = []
all_fr_tokens = []

for example in train_data:
    all_en_tokens.append(example["en_tokens"])
    all_fr_tokens.append(example["fr_tokens"])

en_vocab = Vocabulary(special_tokens=special_tokens, unk_token=unk_token)
en_vocab.build_vocab(all_en_tokens, min_freq=min_freq)

fr_vocab = Vocabulary(special_tokens=special_tokens, unk_token=unk_token)
fr_vocab.build_vocab(all_fr_tokens, min_freq=min_freq)

In [20]:
assert en_vocab[unk_token] == fr_vocab[unk_token]
assert en_vocab[pad_token] == fr_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [21]:
en_vocab.set_default_index(unk_index)
fr_vocab.set_default_index(unk_index)

Checking an example, we can see that it has the two new features: "en_ids" and "de_ids", both a list of integers representing their indices in the respective vocabulary.


In [22]:
train_data[0]

{'en': 'The wind was so strong, we were nearly blown off the road.',
 'fr': 'Le vent était tellement fort que nous avons presque été poussés en dehors de la route.',
 'en_tokens': ['<sos>',
  'the',
  'wind',
  'was',
  'so',
  'strong',
  ',',
  'we',
  'were',
  'nearly',
  'blown',
  'off',
  'the',
  'road',
  '.',
  '<eos>'],
 'fr_tokens': ['<sos>',
  'le',
  'vent',
  'était',
  'tellement',
  'fort',
  'que',
  'nous',
  'avons',
  'presque',
  'été',
  'poussés',
  'en',
  'dehors',
  'de',
  'la',
  'route',
  '.',
  '<eos>']}

We can confirm the indices are correct by using the `lookup_tokens` method with the corresponding vocabulary on the list of indices.


In [23]:
for dataset in [train_data, validation_data, test_data]:
    for example in dataset.examples:
        example["en_ids"] = en_vocab.lookup_indices(example["en_tokens"])
        example["fr_ids"] = fr_vocab.lookup_indices(example["fr_tokens"])

In [24]:
en_vocab.lookup_tokens(train_data[0]["en_ids"])

['<sos>',
 'the',
 'wind',
 'was',
 'so',
 'strong',
 ',',
 'we',
 'were',
 'nearly',
 'blown',
 'off',
 'the',
 'road',
 '.',
 '<eos>']

In [25]:
train_data[0]

{'en': 'The wind was so strong, we were nearly blown off the road.',
 'fr': 'Le vent était tellement fort que nous avons presque été poussés en dehors de la route.',
 'en_tokens': ['<sos>',
  'the',
  'wind',
  'was',
  'so',
  'strong',
  ',',
  'we',
  'were',
  'nearly',
  'blown',
  'off',
  'the',
  'road',
  '.',
  '<eos>'],
 'fr_tokens': ['<sos>',
  'le',
  'vent',
  'était',
  'tellement',
  'fort',
  'que',
  'nous',
  'avons',
  'presque',
  'été',
  'poussés',
  'en',
  'dehors',
  'de',
  'la',
  'route',
  '.',
  '<eos>'],
 'en_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 4, 15, 16, 3],
 'fr_ids': [2,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  3]}

# Data Loaders

In [26]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [torch.tensor(example["en_ids"], dtype=torch.long) for example in batch]
        batch_fr_ids = [torch.tensor(example["fr_ids"], dtype=torch.long) for example in batch]

        # CHANGED: Added batch_first=False (this forces shape to [seq_len, batch_size])
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index, batch_first=False)
        batch_fr_ids = nn.utils.rnn.pad_sequence(batch_fr_ids, padding_value=pad_index, batch_first=False)

        batch = {
            "en_ids": batch_en_ids,
            "fr_ids": batch_fr_ids,
        }
        return batch

    return collate_fn

In [27]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False
    )
    return data_loader

In [28]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(validation_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# Building the Model (2 layer LSTM)

We'll be building our model in three parts. The encoder, the decoder and a seq2seq model that encapsulates the encoder and decoder and will provide a way to interface with each.

## Encoder

In [29]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

## Decoder

Next, we'll build our decoder, which will also be a 2-layer (4 in the paper) LSTM.

In [30]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

## Seq2Seq

In [31]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0, :]
        for t in range(1, trg_length):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

        return outputs

# Training the Model

Now we have our model implemented, we can begin training it.

In [43]:
input_dim = len(en_vocab) # Source language is French
output_dim = len(fr_vocab) # Target language is English
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

## Weight Initialization

We will initialize all weights from a uniform distribution between -0.08 and +0.08.


In [44]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

In [45]:
model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(8219, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(12898, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=12898, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

We can also count the number of parameters in our model.


In [46]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 19,379,042 trainable parameters


## Optimizer

In [47]:
optimizer = optim.Adam(model.parameters())

## Loss Function

In [48]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

In [49]:
TRG_PAD_IDX = fr_vocab.get_stoi()["<pad>"]
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

## Training Loop


In [50]:
def train_fn(model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device):
    model.train()
    epoch_loss = 0
    total_batches = len(data_loader)

    print(f"   [Train] Total batches to process: {total_batches}")

    for i, batch in enumerate(data_loader):
        src = batch["en_ids"].to(device)
        trg = batch["fr_ids"].to(device)

        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)

        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg_target = trg[1:].view(-1)

        loss = criterion(output, trg_target)
        loss.backward()

        #torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

        # FORCE A NEW LINE PRINT EVERY SINGLE BATCH
        print(f"   >>> [Train] Core Engine Active: Completed Batch {i+1}/{total_batches} | Current Loss: {loss.item():.4f}")

    print("\n   [Train] All training batches completed for this epoch.")
    return epoch_loss / total_batches

## Evaluation Loop

In [51]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    print(f"   [Eval] Total batches to process: {len(data_loader)}")

    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["en_ids"].to(device)
            trg = batch["fr_ids"].to(device)

            output = model(src, trg, 0)

            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg_target = trg[1:].view(-1)

            loss = criterion(output, trg_target)
            epoch_loss += loss.item()

    print("[Eval] All evaluation batches completed.")
    return epoch_loss / len(data_loader)

# Model Training

We can finally start training our model!

In [52]:
'''
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")
print(f"Starting execution loop for {n_epochs} Epochs...\n")

for epoch in range(n_epochs):
    start_time = time.time()

    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )

    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(int(end_time - start_time), 60)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
        print(f" -> Epoch {epoch+1}: Discovered lower loss footprint! Weights stored.")

    print(f"Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}\n")
    '''

'\nn_epochs = 10\nclip = 1.0\nteacher_forcing_ratio = 0.5\n\nbest_valid_loss = float("inf")\nprint(f"Starting execution loop for {n_epochs} Epochs...\n")\n\nfor epoch in range(n_epochs):\n    start_time = time.time()\n\n    train_loss = train_fn(\n        model,\n        train_data_loader,\n        optimizer,\n        criterion,\n        clip,\n        teacher_forcing_ratio,\n        device,\n    )\n\n    valid_loss = evaluate_fn(\n        model,\n        valid_data_loader,\n        criterion,\n        device,\n    )\n\n    end_time = time.time()\n    epoch_mins, epoch_secs = divmod(int(end_time - start_time), 60)\n\n    if valid_loss < best_valid_loss:\n        best_valid_loss = valid_loss\n        torch.save(model.state_dict(), "tut1-model.pt")\n        print(f" -> Epoch {epoch+1}: Discovered lower loss footprint! Weights stored.")\n\n    print(f"Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s")\n    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3

# Evaluating the Model


In [53]:
model.load_state_dict(torch.load("tut1-model.pt", map_location=torch.device('cpu')))

test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

   [Eval] Total batches to process: 107
[Eval] All evaluation batches completed.
| Test Loss: 2.699 | Test PPL:  14.867 |


In [80]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    fr_nlp,
    en_vocab,
    fr_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        # Source language for this checkpoint is French
        if isinstance(sentence, str):
            tokens = [token.text for token in fr_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]

        if lower:
            tokens = [token.lower() for token in tokens]

        tokens = [sos_token] + tokens + [eos_token]
        ids = fr_vocab.lookup_indices(tokens)

        # STRICT GUARDRAIL: Must be strictly less than the total rows minus 1
        max_allowed_index = model.encoder.embedding.num_embeddings - 1

        safe_ids = [idx if (0 <= idx < max_allowed_index) else fr_vocab.get_stoi().get("<unk>", 0) for idx in ids]

        tensor = torch.tensor(safe_ids, dtype=torch.long).unsqueeze(-1).to(device)

        # Tutorial 1 Encoder only returns hidden and cell states (LSTM)
        hidden, cell = model.encoder(tensor)

        # Target language for this checkpoint is English
        inputs = en_vocab.lookup_indices([sos_token])

        for _ in range(max_output_length):
            inputs_tensor = torch.tensor([inputs[-1]], dtype=torch.long).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)

            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)

            if predicted_token == en_vocab[eos_token]:
                break

        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [81]:
sentence = test_data[0]["fr"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

("À l'aide\u202f!", 'Help!')

In [82]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    fr_nlp,
    en_vocab,
    fr_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [83]:
translation

['<sos>', 'that', 'you', 'if', 'may', 'go', 'be', '<eos>']

In [84]:
sentence = "I eat the apple."
expected_translation = "Je mange la pomme."

In [85]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    fr_nlp,
    en_vocab,
    fr_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [86]:
translation

['<sos>', 'can', '<unk>', '<unk>', 'you', 'bush', 'be', '<eos>']

In [87]:
translations = [
    translate_sentence(
        example["fr"],
        model,
        en_nlp,
        fr_nlp,
        en_vocab,
        fr_vocab,
        lower,
        sos_token,
        eos_token,
        device,
    )
    for example in tqdm.tqdm(test_data)
]

100%|██████████| 13584/13584 [27:49<00:00,  8.14it/s]


# Metrics (BLEU)


In [88]:
bleu = evaluate.load("bleu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [89]:
predictions = [" ".join(translation[1:-1]) for translation in translations]

references = [[example["en"]] for example in test_data]

In [90]:
predictions[0], references[0]

('that you if may go be', ['Help!'])

In [91]:
def get_tokenizer_fn(nlp, lower):
    def tokenizer_fn(s):
        tokens = [token.text for token in nlp.tokenizer(s)]
        if lower:
            tokens = [token.lower() for token in tokens]
        return tokens

    return tokenizer_fn

In [92]:
tokenizer_fn = get_tokenizer_fn(en_nlp, lower)

In [93]:
tokenizer_fn(predictions[0]), tokenizer_fn(references[0][0])

(['that', 'you', 'if', 'may', 'go', 'be'], ['help', '!'])

In [94]:
results = bleu.compute(
    predictions=predictions, references=references, tokenizer=tokenizer_fn
)

In [95]:
results

{'bleu': 0.0,
 'precisions': [0.03294543473184401, 0.0002489685588277138, 0.0, 0.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.4922899706699448,
 'translation_length': 154164,
 'reference_length': 103307}